# Notebook Information: Test Sets
- Test set 1 = Easy and should cause no problems. Predict Correctly.
- Test set 2 = Should try to "break" our model. Predict Incorrectly.
- Each test set (6 reviews | 3 positive | 3 negative)

I will source my film reviews from a movie review website/app called Letterboxd.\
Letterboxd users rate films on a scale of 1-5 stars. In 0.5 increments\
(0.5*, 1*, 1.5*, 2*, 2.5*, 3*, 3.5*, 4*, 4.5*, 5*)\
I will include some of my own reviews along with some reviews made by other people

## Model

In [1]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

imdb_dataset = load_dataset("imdb")["train"]

train_data = []
train_data_labels = []
for item in imdb_dataset:
    train_data.append(item["text"])
    train_data_labels.append(item["label"])

vectorizer = TfidfVectorizer(analyzer="word", max_features=5000, lowercase=True)
features = vectorizer.fit_transform(train_data).toarray()

x_train, x_val, y_train, y_val = train_test_split(features, train_data_labels, train_size=0.9, random_state=123)
model = MultinomialNB()
model = model.fit(X=x_train, y=y_train)

## Test set 1: Easy - Predict Correctly
Positive Reviews for the film "Dune: Part two"\
Negative Reviews for the film "Megalopolis"

### Why I picked these film reviews
**Positive reviews**\
For my three positive film reviews which should have no problem predicting as positive, i chose reviews for the film "Dune: Part Two".\
Dune: Part Two averages a 4.4 star rating, with the majortity of people rating it a 5* film\
This showed me that I should easily be able to find positive reviews for the film, due to it being so highly rated

**Negative Reviews**\
For my three negative film reviews which should have no problem predicting as negative, I chose reviews for the film "Megalopolis"\
Megalopolis averages a 2.4 star rating, with the majority of people rating it a 2* film\
This showed me that I should easily be able to find negative reviews for the film, due to it being so poorly rated

In [3]:
import pandas as pd
import numpy as np

In [8]:
dune = ["Watching this in IMAX is the correct way to watch. That is what the movies were made for. Incredible",
        "This is truly the cinematic event of our generation… don’t take it for granted",
        "this is #1. my favorite movie. it led me to paradise."]

megalopoplis = ["""Doesn't know what it should be. Cool idea but it feels like the end product doesn't match the vision. 
Some of the cinematography was brilliant while some scenes completely contradict that point. 
As the film progresses the obvious green screen becomes hard to avoid, which probably comes down to budgeting issues. 
Sometimes there's several plots going on at once, with none that are really good enough to really catch you attention, 
making it difficult to understand whats actually happening. 
Positive points are that there are some really nice shots & the world seems interesting to explore and understand. 
Glad to have watched because it's definitely an experience.""",
                "WORST movie i have ever seen.", 
                "most nonsensical and absurd movie i ever saw, unpleasant and annoying"]

dune_df = pd.DataFrame({'text': dune, 'label': [1, 1, 1]}, index=[0, 1, 2])
megalopoplis_df = pd.DataFrame({'text': megalopoplis, 'label': [0, 0, 0]}, index=[3, 4, 5])

test1 = pd.concat([dune_df, megalopoplis_df])

### Positive Dune Reviews:
1) Looks like a positive review from reading. Contains positive wor "Incredible" and somewhat positive word "correct". Does not seem to contain any negative words. Should predict as positive.
2) A positive review. Contains positive word: "Truly". Has a negative word used positively: "don't". However i feel that "truly" will overpower "don't" in predcitions.
3) Clearly a positive review. Line saying: "This is my favourite movie". Positive word "favourite" and somewhat positive word "paradise". Does not seem to contain any negative words. Should predict as positive.

### Negative Megalopolis Reviews
1) Negative review. Contains positive and negative elements but I feel the negative definitely outweighs the positve, or is stronger. Negative words: "doesn't", "avoid", "issues", "difficult". Positive words: "glad", "brilliant", "nice".
2) Clearly negative. Contains highly negative word: "worst", with the review simply being: "Worst movie i have ever seen". Should have no problem predicting negatively
3) Clearly negative. Contains only negative words: "nonsensical", "absurd", "unpleasant", "annoying". Should again have no issue predicting negatively

In [62]:
from sklearn.metrics import accuracy_score

x_test1 = test1["text"]
y_test1 = test1["label"]

y_pred1 = model.predict(vectorizer.transform(x_test1).toarray())
test1["predictions"] = y_pred1
print(test1)
print(accuracy_score(y_test1, y_pred1))

                                                text  label  predictions
0  Watching this in IMAX is the correct way to wa...      1            1
1  This is truly the cinematic event of our gener...      1            1
2  this is #1. my favorite movie. it led me to pa...      1            1
3  Doesn't know what it should be. Cool idea but ...      0            0
4                      WORST movie i have ever seen.      0            0
5  most nonsensical and absurd movie i ever saw, ...      0            0
1.0


Model has 100% Accuracy. Predicts all of the reviews correctly.

## Test set 2: Break the Model - Predict Incorrectly
Positive Reviews for the film "Layer Cake"\
Negative Reviews for the film "Jack and Jill"

### Why I picked these film reviews

**Positive**\
For my three positive reviews which should predict incorrectly, I chose the film "Layer Cake"\
Layer Cake averages a 3.5 star rating, with the majority of people rating it between 2.5-3.5 stars i.e. averagely\
This means that I should hopefuly be able to find some positive reviews which contain negative elements in order to confuse my model and predict negatively, as this film is not as highly rated as Dune: Part Two

**Negative**\
For my three negative reviews which should predict incorrectly, I chose the film "Jack and Jill"\
Jack and Jill averages a 1.4 star rating, with the majority of people rating it a 0.5 star film\
As this is a comedy movie I hope to find witty, or confusing reviews of the film which while negative should confuse my model and predict positive.

In [59]:
layer_cake = ["Action movie with a plot that keeps you there, confused but decent nonetheless", 
            "The budget of Layer Cake was 6.5 million. It looked like it was shot for 60 million dollars.", 
            "that yellow range rover is so ugly, yet so cool"]

jack_and_jill = ["Adam Sandler has blackmail on all of Hollywood, doesn't he", 
                 "I enjoyed none of this", 
                 "I thank my lucky stars I was unconscious for most of this"]

layer_cake_df = pd.DataFrame({'text': layer_cake, 'label': [1, 1, 1]}, index=[0, 1, 2])
jack_and_jill_df = pd.DataFrame({'text': jack_and_jill, 'label': [0, 0, 0]}, index=[3, 4, 5])

test2 = pd.concat([layer_cake_df, jack_and_jill_df])

### Positive Layer Cake reviews
1) Positive review which contains negative and positive words. Positive: "decent", Negative: "confused". Confused appears before engaged and might have a greater weight, leading the model to predict this as a negative review.
2) This review is praising the film, saying it looks more expensive than it is, meaning it exceeds expectations. The review does not contain any true positive or negative words. However I believe the word "budget" may be used negatively in the context of film reviews, leading to a negative prediction. More often than not I would say that when people mention budget in a film review they are using it to say how the film looks bad on such a high budget etc.
3) Contains positive and negative words, however I feel the negative word may outweigh the positve. Negative word: "ugly", Positive word: "cool".

### Negative Jack and Jill reviews
1) Contains no real positive or negative words, while it clearly represents a negative review. The word "Hollywood" may represent a positive word as in reviews users may typically use Hollywood to highlight a films professional quality
2) Contains positive and negative words. However the positive words feels as though it outweighs the negative word. Positive word: "enjoyed", Negative word: "none"
3) Negative review which features positive words: "thank", "lucky". Does not contain any clear negative words while being a negative review. The term "unconscious" means this is a bad review, but I don't believe it will have a great weight even if its said to be a negative word.

In [61]:
x_test2 = test2["text"]
y_test2 = test2["label"]

y_pred2 = model.predict(vectorizer.transform(x_test2).toarray())
test2["predictions"] = y_pred2
print(test2)
print(accuracy_score(y_test2, y_pred2))

                                                text  label  predictions
0  Action movie with a plot that keeps you there,...      1            0
1  The budget of Layer Cake was 6.5 million. It l...      1            0
2    that yellow range rover is so ugly, yet so cool      1            0
3  Adam Sandler has blackmail on all of Hollywood...      0            1
4                             I enjoyed none of this      0            1
5  I thank my lucky stars I was unconscious for m...      0            1
0.0


Model as 0% Accuracy. Predicts all of the reviews incorrectly